In [4]:
import pandas as pd
import numpy as np
import os

os.makedirs("data/processed", exist_ok=True)

In [5]:
nav = pd.read_csv("../data/processed/nav_history_cleaned.csv")
nav["date"] = pd.to_datetime(nav["date"])
nav = nav.sort_values(["amfi_code", "date"]).reset_index(drop=True)

In [6]:
# Compute daily return: nav_t / nav_t-1 - 1
nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()

In [7]:
# Drop the first row of each scheme (NaN return)
nav_clean = nav.dropna(subset=["daily_return"]).copy()

In [8]:
print("\nDaily Return Distribution (all schemes):")
print(nav_clean["daily_return"].describe())


Daily Return Distribution (all schemes):
count    45960.000000
mean         0.000631
std          0.010290
min         -0.058102
25%         -0.005042
50%          0.000340
75%          0.006324
max          0.064713
Name: daily_return, dtype: float64


In [9]:
print("\nExtreme daily returns (>|5%|):")
extreme = nav_clean[nav_clean["daily_return"].abs() > 0.05]
print(f"Count: {len(extreme)}")
if len(extreme) > 0:
    print(extreme.nlargest(5, "daily_return")[["amfi_code", "date", "nav", "daily_return"]])
    print(extreme.nsmallest(5, "daily_return")[["amfi_code", "date", "nav", "daily_return"]])


Extreme daily returns (>|5%|):
Count: 9
       amfi_code       date       nav  daily_return
24745     119598 2024-04-15  189.0742      0.064713
13226     118634 2024-03-19  110.5401      0.059304
5207      101207 2024-05-01   65.5882      0.054851
25566     119599 2023-01-10  153.2221      0.053320
25832     119599 2024-01-17  136.8239      0.051811
       amfi_code       date       nav  daily_return
12774     118634 2022-06-24   67.2534     -0.058102
4917      101207 2023-03-22   56.8326     -0.051847
12952     118634 2023-03-01  113.5801     -0.050335
24155     119598 2022-01-10   96.0964      0.051113
25832     119599 2024-01-17  136.8239      0.051811


In [12]:
output_path = "../data/processed/daily_returns.csv"
nav_clean.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")


Saved to: ../data/processed/daily_returns.csv
